# 08 - ReactionT5 + Conditions Agent Inference Demo

Runs `product -> reactants -> conditions` and converts the output into the app-compatible route JSON schema.

- **Reactants**: [`sagawa/ReactionT5v2-retrosynthesis`](https://huggingface.co/sagawa/ReactionT5v2-retrosynthesis), the retrosynthesis checkpoint from Sagawa & Kojima, *"ReactionT5: a pre-trained transformer model for accurate chemical reaction prediction with limited data"*, J. Cheminform. 17:126 (2025). Per the paper, retrosynthesis prediction takes the raw product SMILES with **no special prefix token** (unlike product/yield prediction, which prepend `REACTANT:` / `REAGENT:` / `PRODUCT:`) and decodes with beam search (Top-5 accuracy 86.9%, Top-5 invalidity 0.45% on USPTO_50k). It's a small T5 (12+12 layers, d_model 768, ~60M params, custom 268-token SMILES vocab), so it's cheap to run alongside the conditions model.
- **Conditions**: same `oleh13/retro-conditions-qwen25-7b-lora` LoRA (on `Qwen/Qwen2.5-7B-Instruct`) used in `06_agent_inference_demo.ipynb`.

ReactionT5 does not output a reaction class, but the conditions model expects one -- a placeholder (`UNKNOWN_REACTION_CLASS`) is passed through instead. If condition quality suffers because of this, consider adding a reaction-classification step upstream.

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes safetensors sentencepiece rdkit

import json
import re
from typing import Optional

import torch
from peft import PeftModel
from rdkit import Chem, RDLogger
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

# Stage 1: reactants, via ReactionT5 (Sagawa & Kojima, 2025; https://doi.org/10.1186/s13321-025-01075-4)
REACTANTS_MODEL_ID = 'sagawa/ReactionT5v2-retrosynthesis'

# Stage 2: conditions, same Qwen LoRA as 06_agent_inference_demo.ipynb
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
CONDITION_LORA_ID = 'oleh13/retro-conditions-qwen25-7b-lora'

CONDITION_SYSTEM = '''You are a chemistry condition recommendation model.
Return valid compact JSON only.
Given product SMILES, reactants, and reaction class, predict practical reaction conditions.
Do not invent evidence IDs and do not include explanations.'''

# ReactionT5 doesn't predict a reaction class; the conditions model was trained
# expecting one, so this placeholder is passed through in its place.
UNKNOWN_REACTION_CLASS = 'unclassified'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 15.2 MB/s eta 0:00:00


In [2]:
def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None
    except Exception:
        return None

def split_reactants(text: str) -> list[str]:
    # Multi-reactant SMILES are '.'-separated on one line (ReactionT5 convention).
    return [p.strip() for p in text.strip().split('.') if p.strip()]

def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text, flags=re.I).strip()
    text = re.sub(r'```$', '', text).strip()
    first, last = text.find('{'), text.rfind('}')
    if first == -1 or last == -1 or last <= first:
        raise ValueError(f'No JSON object found: {text[:300]}')
    return json.loads(text[first:last + 1])

def load_reactants_model():
    tok = AutoTokenizer.from_pretrained(REACTANTS_MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(REACTANTS_MODEL_ID, dtype=torch.float32)
    model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    return tok, model

def load_condition_model():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto', dtype=torch.bfloat16, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, CONDITION_LORA_ID)
    model.eval()
    return tok, model


In [3]:
def generate_reactant_candidates(tok, model, product_smiles: str, num_beams=5, num_return_sequences=5, max_length=150) -> list[str]:
    # No task prefix: retrosynthesis prediction takes the raw product SMILES (Sagawa & Kojima, 2025, Sect. 'Retrosynthesis prediction').
    inputs = tok([product_smiles], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            num_beams=num_beams,
            num_return_sequences=min(num_return_sequences, num_beams),
            max_length=max_length,
            min_length=1,
        )
    decoded = tok.batch_decode(out, skip_special_tokens=True)
    # ReactionT5's tokenizer emits space-separated tokens; join before canonicalizing.
    return [d.strip().replace(' ', '') for d in decoded]

def generate_condition_json(tok, model, messages, max_new_tokens=320):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([prompt], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    raw = tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    return extract_json(raw), raw


In [4]:
react_tok, react_model = load_reactants_model()
cond_tok, cond_model = load_condition_model()

def predict_reactants(product_smiles: str, num_beams=5, num_return_sequences=5) -> list[list[str]]:
    target = canonical_smiles(product_smiles)
    if not target:
        raise ValueError(f'Invalid target SMILES: {product_smiles}')
    raw_candidates = generate_reactant_candidates(react_tok, react_model, target, num_beams=num_beams, num_return_sequences=num_return_sequences)
    reactant_sets, seen = [], set()
    for raw in raw_candidates:
        reactants = [canonical_smiles(p) for p in split_reactants(raw)]
        reactants = [r for r in reactants if r]
        key = tuple(sorted(reactants))
        if reactants and key not in seen:
            seen.add(key)
            reactant_sets.append(reactants)
    if not reactant_sets:
        raise ValueError(f'No valid reactants predicted. Raw candidates: {raw_candidates}')
    return reactant_sets

def predict_conditions(product_smiles: str, reactants: list[str], reaction_class: str = UNKNOWN_REACTION_CLASS) -> dict:
    user = json.dumps({'task': 'predict_conditions', 'product_smiles': product_smiles, 'reactants': reactants, 'reaction_class': reaction_class}, separators=(',', ':'))
    data, _raw = generate_condition_json(cond_tok, cond_model, [{'role': 'system', 'content': CONDITION_SYSTEM}, {'role': 'user', 'content': user}], max_new_tokens=320)
    return data


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/36.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/795M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

In [5]:
def to_app_route(product_smiles: str, routes_data: list[dict]) -> dict:
    routes = []
    for i, rd in enumerate(routes_data):
        reactants = rd['reactants']
        condition_result = rd['condition_result']
        routes.append({
            'route_name': f'ReactionT5 candidate {i + 1}',
            'summary': 'ReactionT5 retrosynthesis proposal with Qwen-predicted conditions.',
            'steps': [{
                'reaction_name': 'predicted single-step retrosynthesis',
                'reactants': reactants,
                'product_smiles': product_smiles,
                'stoichiometry': 'Reactants as predicted; optimize equivalents experimentally.',
                'reagents': condition_result.get('reagents', 'not specified'),
                'solvent': condition_result.get('solvent', 'not specified'),
                'temperature_celsius': condition_result.get('temperature_celsius', 'not specified'),
                'reaction_time': condition_result.get('reaction_time', 'not specified'),
                'atmosphere': condition_result.get('atmosphere', 'not specified'),
                'workup_purification': condition_result.get('workup_purification', 'not specified'),
                'expected_yield_percent': condition_result.get('expected_yield_percent', 'not specified'),
                'important_conditions': 'Verify route experimentally; model output is a prediction.',
                'rationale': 'Reactants were generated by ReactionT5 (Sagawa & Kojima, 2025); conditions were generated by a separately fine-tuned Qwen LoRA model.',
                'objective_fit': 'Single-step route proposal from target SMILES.',
                'evidence_reaction_ids': [],
            }],
            'objective_fit': 'Single-step route proposal from target SMILES.',
            'evidence_reaction_ids': [],
        })
    return {
        'routes': routes,
        'overall_recommendation': 'Use as candidate routes and validate with literature or experiment.',
    }

def predict_route(product_smiles: str, num_routes: int = 1, num_beams: int = 5, num_return_sequences: int = 5) -> dict:
    target = canonical_smiles(product_smiles)
    reactant_sets = predict_reactants(target, num_beams=num_beams, num_return_sequences=num_return_sequences)
    routes_data = []
    for reactants in reactant_sets[:max(1, num_routes)]:
        condition_result = predict_conditions(target, reactants)
        routes_data.append({'reactants': reactants, 'condition_result': condition_result})
    return to_app_route(target, routes_data)


In [6]:
test_targets = [
    'N#CCc1ccccc1',
    'O=[N+]([O-])c1ccccc1',
    'CC(C)(O)c1ccccc1',
    'c1ccc(-c2ccccc2)cc1',
    'O=C(/C=C/c1ccccc1)c1ccccc1',
    'C=Cc1ccccc1',
    'C1=CCCCC1',
    'CC(O)c1ccccc1',
    'O=C1OC(=O)C2CC=CCC12',
    'CCNC(C)=O',
]

for target in test_targets:
    print('\nTARGET:', target)
    try:
        result = predict_route(target)
        print(json.dumps(result, indent=2, ensure_ascii=False)[:4000])
    except Exception as exc:
        print('FAILED GRACEFULLY:', exc)



TARGET: N#CCc1ccccc1


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "routes": [
    {
      "route_name": "ReactionT5 candidate 1",
      "summary": "ReactionT5 retrosynthesis proposal with Qwen-predicted conditions.",
      "steps": [
        {
          "reaction_name": "predicted single-step retrosynthesis",
          "reactants": [
            "BrCc1ccccc1",
            "[C-]#N",
            "[Na+]"
          ],
          "product_smiles": "N#CCc1ccccc1",
          "stoichiometry": "Reactants as predicted; optimize equivalents experimentally.",
          "reagents": "not specified",
          "solvent": "water; acetonitrile",
          "temperature_celsius": "not specified",
          "reaction_time": "not specified",
          "atmosphere": "not specified",
          "workup_purification": "TEMPERATURE: The mixture was heated; TEMPERATURE: at reflux for 2 hours; CUSTOM: The organic layer was separated; WASH: washed with water (3×50 mL); DRY_WITH_MATERIAL: dried over MgSO4; FILTRATION: filtered; CONCENTRATION: concentrated in vacuo",
          